# 04 — Train: XGBoost regression

Searches an XGBoost gradient-boosted-tree hyperparameter configuration for the configured target station's direct multi-horizon water-level forecast over the joined feature artifacts, then evaluates the selected model once on the sealed test cohort.

**Inputs:** joined train/test feature artifacts and their metadata contract
**Outputs:** in-notebook prediction preview/test metrics, an MLflow run hierarchy, and the selected model plus manifest in `models/`

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports the dependencies and loads the joined feature metadata. Its predictor columns are the source of truth for the model inputs: target-station engineered features plus raw measurements from every retained station at issue time `t`. The hyperparameter search and validation policy are explicit constants so every fold and MLflow run remains inspectable.

Unlike Ridge and MLP, `XGBRegressor` fits decision trees, which are invariant to monotonic feature transforms and need no predictor scaling — there is no log1p axis and no preprocessing pipeline; `build_xgboost_estimator` returns a bare `XGBRegressor`. `RANDOM_STATE` is still fixed and reused in every fold fit and the final retrain (it seeds row/column subsampling), so the manifest remains a reproducible record. Multi-output fitting uses `multi_strategy="one_output_per_tree"`, so one call fits every metadata-declared horizon together, matching Ridge/MLP's single-call multi-output interface.

The native XGBoost hyperparameter space is large, so rather than an exhaustive grid like Ridge/MLP, `XGBOOST_N_ITER` combinations are drawn once via `sklearn.model_selection.ParameterSampler` with the fixed `RANDOM_STATE`, and the identical sampled set is reused for every feature subset — a fair, comparable search at a fraction of the exhaustive grid's cost.

**Parameters**

The values below are illustrative examples, not run configuration. Imported values from `src/config.py` and executable constants in this notebook are authoritative for a run; Stage-3 feature metadata is authoritative for the realized column contract, and the saved manifest records the completed execution.

| Parameter | Example value | What it does |
| --- | --- | --- |
| `PROCESSED_DIR` | `data/processed/joined` | Directory the joined Stage-3 Parquets and metadata are read from. |
| `PREDICTION_PREVIEW_ROWS` | `5` | Number of scored test rows shown in the final preview. |
| `FULL_FEATURE_COLUMNS` | all metadata-declared predictors | The complete predictor contract used for common eligibility; raw timestamps and metadata fields are not model inputs. |
| `FEATURE_SUBSETS` | named derived subsets | Candidate feature lists built at runtime from the full metadata contract in metadata order. |
| `TARGET_COLUMNS` | `{TARGET_STATION_ID}__target_t_plus_01` … `{TARGET_STATION_ID}__target_t_plus_H` | The metadata-declared future water levels predicted directly from one issue-time feature vector. |
| `FORECAST_HORIZON_HOURS` | `24` | Example configured horizon; execution uses the imported value and validates it against the metadata contract width. |
| `XGBOOST_PARAM_DISTRIBUTIONS` | see below | Discrete candidate values per hyperparameter, sampled by `ParameterSampler`. |
| `XGBOOST_N_ITER` | `32` | Number of hyperparameter combinations sampled per feature subset. |
| `XGBOOST_N_JOBS` | `-1` | Threads used internally by each `XGBRegressor` fit; the outer subset/hyperparameter/fold loop stays sequential. |
| `RANDOM_STATE` | `src.config.RANDOM_STATE` | Fixed seed for the hyperparameter sampler and every fold fit/final retrain's row/column subsampling. |
| `N_VALIDATION_FOLDS` | `5` | Number of expanding-window validation folds. |
| `INITIAL_TRAIN_FRACTION` | `0.50` | Approximate fraction of eligible rows in the first fold's training window. |
| `EMBARGO_HOURS` | `24` | Number of rows left between each fold's training and validation windows. |
| `CV_SELECTION_METRIC` | `"rmse"` | Aggregate CV metric used to select the candidate; `"mae"` is also supported. |
| `MLFLOW_EXPERIMENT_NAME` | `"xgboost"` | Experiment receiving the parent, nested fold, and final test runs. |
| `MODEL_PATH` | `models/xgboost_{TARGET_STATION_ID}.joblib` | Selected `XGBRegressor` trained on all eligible training rows. |
| `MODEL_METADATA_PATH` | `models/xgboost_{TARGET_STATION_ID}.json` | Schema-1.0 manifest recording the feature contract, selected subset/hyperparameters, the full CV candidate table, the aggregate and per-horizon sealed-test metrics, and the training range. Written once, after the sealed test. |

Illustrative hyperparameter candidate values (`XGBOOST_PARAM_DISTRIBUTIONS` in the executable cell is authoritative):

| Hyperparameter | Example candidate values |
| --- | --- |
| `max_depth` | `3, 4, 5, 6, 7` |
| `learning_rate` | `0.01, 0.03, 0.05, 0.1, 0.2, 0.3` |
| `n_estimators` | `100, 200, 300, 500` |
| `subsample` | `0.6, 0.8, 1.0` |
| `colsample_bytree` | `0.6, 0.8, 1.0` |
| `reg_lambda` | `0.1, 1.0, 10.0` |

## Joint feature-subset and hyperparameter search

The notebook compares `(feature subset, hyperparameter combination)` pairs across `N_VALIDATION_FOLDS` expanding-window folds. The named subsets are the same ones used by `04_02_train_ridge.ipynb` and `04_03_train_mlp.ipynb`, derived from metadata-declared predictors and retaining their metadata order:

| Subset | Intended predictors | Example size |
| --- | --- | ---: |
| `full` | All declared predictors | 81 |
| `all_station_hydrology_quality_time` | Water-level history, imputation indicators, and calendar signals for every station; excludes weather | 55 |
| `raw_all_stations` | Current `water_level`, `imputed`, precipitation, and temperature for every station | 32 |
| `target_station_full` | All declared predictors for the target station only | 53 |
| `target_station_hydrology_quality_time` | Target-station water-level history, imputation indicators, and calendar signals | 41 |
| `current_water_levels_all_stations` | Current `water_level` for every station | 8 |

The example sizes above come from one feature contract and are non-authoritative; `FEATURE_SUBSETS` built by `load_joined_dataset()` supplies the realized columns and sizes. The full contract determines eligibility once for both artifacts, so all candidates use the same eligible rows and identical fold indices. `XGBOOST_N_ITER` hyperparameter combinations are sampled once (seeded by `RANDOM_STATE`) and reused identically for every subset, so the search performs `len(FEATURE_SUBSETS) × XGBOOST_N_ITER × N_VALIDATION_FOLDS` fold fits, then retrains only the selected candidate and evaluates the sealed test once.

In [ ]:
import json
from pathlib import Path
from uuid import uuid4

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import dump
from sklearn.model_selection import ParameterSampler  # type: ignore[import-untyped]
from tqdm.auto import tqdm

from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    MLFLOW_TRACKING_URI,
    N_VALIDATION_FOLDS,
    RANDOM_STATE,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.dataset import load_joined_dataset
from src.metrics import metric_tables
from src.plots import (
    cv_error_boxplots_figure,
    predicted_vs_actual_figure,
    test_error_boxplots_figure,
)
from src.training import (
    numeric_predictors,
    prediction_preview,
    summarize_cv_metrics,
    validate_predictions,
)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
NOTEBOOK_EXECUTION_UUID = str(uuid4())
PROCESSED_DIR = Path("data/processed/joined")
METADATA_PATH = PROCESSED_DIR / "all_stations_feature_metadata.json"
train_path = PROCESSED_DIR / "all_stations_train_features.parquet"
test_path = PROCESSED_DIR / "all_stations_test_features.parquet"
XGBOOST_PARAM_DISTRIBUTIONS = {
    "max_depth": [3, 4, 5, 6, 7],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2, 0.3],
    "n_estimators": [100, 200, 300, 500],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_lambda": [0.1, 1.0, 10.0],
}
XGBOOST_N_ITER = 32
XGBOOST_N_JOBS = -1
MLFLOW_EXPERIMENT_NAME = "xgboost"
PREDICTION_PREVIEW_ROWS = 5
if CV_SELECTION_METRIC not in {"mae", "rmse"}:
    raise ValueError("CV_SELECTION_METRIC must be either 'mae' or 'rmse'")
station_id = TARGET_STATION_ID
MODEL_DIR = Path("models")
MODEL_PATH = MODEL_DIR / f"xgboost_{station_id}.joblib"
MODEL_METADATA_PATH = MODEL_DIR / f"xgboost_{station_id}.json"

## Shared evaluation cohort

The model is fit and scored on rows from the joined feature artifacts. One row is one timestamp `t`, and it qualifies only when both conditions hold:

1. **Stage 3 marked the future window valid.** `{TARGET_STATION_ID}__target_valid` is true, and every column in `TARGET_COLUMNS` is present.
2. **Every model input is present.** All full-contract predictors must be available: the target station's engineered features plus every retained station's raw water level, imputation flag, precipitation, and temperature at issue time `t`.

Train and test are filtered independently and are never pooled: the test artifact is sealed, and no statistic used by the model is ever computed from it. Eligibility is deliberately based on `FULL_FEATURE_COLUMNS`, not a candidate subset, so all candidates compare the same cohort.

## Shared helpers

The joined dataset — contract loading, common-cohort preparation, ordered feature subsets, and chronological folds — comes from `src.dataset`. Prediction checks, metric summaries, and previews come from `src.training`, and the evaluation figures from `src.plots`. XGBoost candidate ranking remains local because its subset/hyperparameter tie-breaking policy is estimator-specific.

In [ ]:
from src.xgboost_model import (
    build_xgboost_estimator,
    save_xgboost_manifest,
    select_candidate,
)

## Load the joined dataset

`load_joined_dataset()` does the whole preamble in one call: it reads the joined feature metadata, `all_stations_train_features.parquet`, and `all_stations_test_features.parquet` from the Stage-3 directory and checks their station, horizon, and column contracts, so a missing or incompatible artifact fails before any model work begins.

It then applies the eligibility cohort to each artifact independently — keeping only target-valid rows with complete predictors and targets, sorted chronologically — derives the named, ordered feature subsets, and builds the configured expanding validation folds. If either split has no eligible row, or the folds violate the configured CV policy, the notebook stops here rather than fitting on an empty frame or reporting a metric computed from nothing.

In [ ]:
dataset = load_joined_dataset(
    METADATA_PATH,
    train_path,
    test_path,
    station_id=station_id,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
contract = dataset.contract
TARGET_COLUMNS = list(contract.target_columns)
FULL_FEATURE_COLUMNS = list(contract.predictor_columns)
FEATURE_SUBSETS = dataset.feature_subsets
train_rows = dataset.train_rows
test_rows = dataset.test_rows
INPUT_PARQUET_SHA256_PARAMS = dataset.input_hashes

## Joint time-series subset and hyperparameter search

Eligible training rows are sorted by issue time before `TimeSeriesSplit` creates `N_VALIDATION_FOLDS` expanding-window folds. The explicit `test_size` allocates the post-initial-training portion across the folds, while `EMBARGO_HOURS` supplies the hourly embargo. `XGBOOST_N_ITER` hyperparameter combinations are sampled once via `ParameterSampler(XGBOOST_PARAM_DISTRIBUTIONS, n_iter=XGBOOST_N_ITER, random_state=RANDOM_STATE)` and the identical sampled set is reused for every feature subset. Each `(subset, hyperparameters)` candidate has one MLflow parent and each fold has one nested child run. The execution must produce `len(FEATURE_SUBSETS) × XGBOOST_N_ITER` candidates before selection, for that candidate count multiplied by `N_VALIDATION_FOLDS` fold fits.

Every fold builds its own `XGBRegressor` via `build_xgboost_estimator` — no preprocessing pipeline, since trees need no predictor scaling — using only that fold's training rows and the candidate's explicit columns, and the fixed `RANDOM_STATE` so every fold's row/column subsampling is reproducible. The sealed test cohort is not referenced until the final fit below.

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
cv_splits = dataset.folds
validation_test_size = dataset.validation_test_size
cv_results_rows = []
cv_horizon_rows_by_candidate = {}
sampled_hyperparameters = list(
    ParameterSampler(
        XGBOOST_PARAM_DISTRIBUTIONS, n_iter=XGBOOST_N_ITER, random_state=RANDOM_STATE
    )
)
expected_candidate_keys = {
    (
        subset_name,
        int(params["max_depth"]),
        int(params["n_estimators"]),
        float(params["learning_rate"]),
        float(params["subsample"]),
        float(params["colsample_bytree"]),
        float(params["reg_lambda"]),
    )
    for subset_name in FEATURE_SUBSETS
    for params in sampled_hyperparameters
}
total_fold_fits = len(expected_candidate_keys) * N_VALIDATION_FOLDS
search_progress = tqdm(total=total_fold_fits, desc="XGBoost CV search", unit="fit")

for subset_name, feature_columns in FEATURE_SUBSETS.items():
    for params in sampled_hyperparameters:
        max_depth = int(params["max_depth"])
        learning_rate = float(params["learning_rate"])
        n_estimators = int(params["n_estimators"])
        subsample = float(params["subsample"])
        colsample_bytree = float(params["colsample_bytree"])
        reg_lambda = float(params["reg_lambda"])
        hyperparameter_label = (
            f"md{max_depth}_lr{learning_rate:g}_ne{n_estimators}_"
            f"ss{subsample:g}_cs{colsample_bytree:g}_rl{reg_lambda:g}"
        )
        fold_aggregate_rows = []
        fold_horizon_rows = []
        with mlflow.start_run(
            run_name=f"xgboost_cv_{subset_name}_{hyperparameter_label}",
            nested=False,
            tags={
                "phase": "cv",
                "run_type": "candidate_parent",
                "subset": subset_name,
                "execution_uuid": NOTEBOOK_EXECUTION_UUID,
            },
        ):
            mlflow.log_params(
                {
                    "phase": "cv",
                    "run_type": "candidate_parent",
                    "subset": subset_name,
                    "feature_count": len(feature_columns),
                    "feature_columns": json.dumps(feature_columns),
                    **INPUT_PARQUET_SHA256_PARAMS,
                    "max_depth": max_depth,
                    "learning_rate": learning_rate,
                    "n_estimators": n_estimators,
                    "subsample": subsample,
                    "colsample_bytree": colsample_bytree,
                    "reg_lambda": reg_lambda,
                    "tree_method": "hist",
                    "multi_strategy": "one_output_per_tree",
                    "n_jobs": XGBOOST_N_JOBS,
                    "random_state": RANDOM_STATE,
                    "n_validation_folds": N_VALIDATION_FOLDS,
                    "validation_test_size": validation_test_size,
                    "embargo_hours": EMBARGO_HOURS,
                    "selection_metric": CV_SELECTION_METRIC,
                    "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                    "common_train_rows": len(train_rows),
                }
            )

            for fold_number, (
                fold_train_indices,
                fold_validation_indices,
            ) in enumerate(cv_splits, start=1):
                with mlflow.start_run(
                    run_name=f"xgboost_cv_{subset_name}_{hyperparameter_label}_fold_{fold_number}",
                    nested=True,
                    tags={
                        "phase": "cv",
                        "run_type": "fold",
                        "subset": subset_name,
                        "fold": str(fold_number),
                        "execution_uuid": NOTEBOOK_EXECUTION_UUID,
                    },
                ):
                    search_progress.set_description(
                        f"XGBoost CV search ({subset_name}, {hyperparameter_label}, fold {fold_number})"
                    )
                    fold_train_rows = train_rows.iloc[fold_train_indices]
                    fold_validation_rows = train_rows.iloc[fold_validation_indices]
                    fold_model = build_xgboost_estimator(
                        max_depth=max_depth,
                        learning_rate=learning_rate,
                        n_estimators=n_estimators,
                        subsample=subsample,
                        colsample_bytree=colsample_bytree,
                        reg_lambda=reg_lambda,
                        n_jobs=XGBOOST_N_JOBS,
                        random_state=RANDOM_STATE,
                    )
                    fold_model.fit(
                        numeric_predictors(fold_train_rows, feature_columns),
                        fold_train_rows[TARGET_COLUMNS],
                    )
                    fold_predictions = validate_predictions(
                        fold_model.predict(
                            numeric_predictors(fold_validation_rows, feature_columns)
                        ),
                        expected_rows=len(fold_validation_rows),
                        target_columns=TARGET_COLUMNS,
                        artifact_name="fold",
                    )

                    fold_aggregate, fold_per_horizon = metric_tables(
                        fold_validation_rows[TARGET_COLUMNS],
                        fold_predictions,
                        target_columns=TARGET_COLUMNS,
                        station_id=station_id,
                    )
                    fold_aggregate_rows.append(fold_aggregate.iloc[0])
                    fold_horizon_rows.append(fold_per_horizon)
                    search_progress.update(1)
                    mlflow.log_params(
                        {
                            "phase": "cv",
                            "run_type": "fold",
                            "subset": subset_name,
                            "feature_count": len(feature_columns),
                            **INPUT_PARQUET_SHA256_PARAMS,
                            "max_depth": max_depth,
                            "learning_rate": learning_rate,
                            "n_estimators": n_estimators,
                            "subsample": subsample,
                            "colsample_bytree": colsample_bytree,
                            "reg_lambda": reg_lambda,
                            "fold": fold_number,
                            "train_rows": len(fold_train_rows),
                            "validation_rows": len(fold_validation_rows),
                            "gap_rows": EMBARGO_HOURS,
                            "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                            "train_start": fold_train_rows["timestamp"]
                            .iloc[0]
                            .isoformat(),
                            "train_end": fold_train_rows["timestamp"]
                            .iloc[-1]
                            .isoformat(),
                            "validation_start": fold_validation_rows["timestamp"]
                            .iloc[0]
                            .isoformat(),
                            "validation_end": fold_validation_rows["timestamp"]
                            .iloc[-1]
                            .isoformat(),
                            "train_index_start": int(fold_train_indices[0]),
                            "train_index_end": int(fold_train_indices[-1]),
                            "validation_index_start": int(fold_validation_indices[0]),
                            "validation_index_end": int(fold_validation_indices[-1]),
                        }
                    )
                    mlflow.log_metrics(
                        {
                            "fold_mae": float(fold_aggregate.iloc[0]["mae"]),
                            "fold_rmse": float(fold_aggregate.iloc[0]["rmse"]),
                            "fold_me": float(fold_aggregate.iloc[0]["me"]),
                            "fold_r2": float(fold_aggregate.iloc[0]["r2"]),
                            **{
                                f"fold_mae_horizon_{row.horizon_hours:02d}": float(
                                    row.mae
                                )
                                for row in fold_per_horizon.itertuples()
                            },
                            **{
                                f"fold_me_horizon_{row.horizon_hours:02d}": float(
                                    row.me
                                )
                                for row in fold_per_horizon.itertuples()
                            },
                            **{
                                f"fold_r2_horizon_{row.horizon_hours:02d}": float(
                                    row.r2
                                )
                                for row in fold_per_horizon.itertuples()
                            },
                            **{
                                f"fold_rmse_horizon_{row.horizon_hours:02d}": float(
                                    row.rmse
                                )
                                for row in fold_per_horizon.itertuples()
                            },
                        }
                    )

            fold_aggregate_metrics = pd.DataFrame(fold_aggregate_rows)
            fold_horizon_metrics = pd.concat(fold_horizon_rows, ignore_index=True)
            parent_metrics = summarize_cv_metrics(
                fold_aggregate_metrics,
                fold_horizon_metrics,
            )
            candidate_key = (
                subset_name,
                max_depth,
                n_estimators,
                learning_rate,
                subsample,
                colsample_bytree,
                reg_lambda,
            )
            cv_horizon_rows_by_candidate[candidate_key] = fold_horizon_rows.copy()
            mlflow.log_metrics(parent_metrics)
            cv_results_rows.append(
                {
                    "subset": subset_name,
                    "feature_count": len(feature_columns),
                    "max_depth": max_depth,
                    "n_estimators": n_estimators,
                    "learning_rate": learning_rate,
                    "subsample": subsample,
                    "colsample_bytree": colsample_bytree,
                    "reg_lambda": reg_lambda,
                    "mae_mean": parent_metrics["cv_mae_mean"],
                    "mae_std": parent_metrics["cv_mae_std"],
                    "rmse_mean": parent_metrics["cv_rmse_mean"],
                    "rmse_std": parent_metrics["cv_rmse_std"],
                    "me_mean": parent_metrics["cv_me_mean"],
                    "me_std": parent_metrics["cv_me_std"],
                    "r2_mean": parent_metrics["cv_r2_mean"],
                    "r2_std": parent_metrics["cv_r2_std"],
                    **{
                        metric_name: metric_value
                        for metric_name, metric_value in parent_metrics.items()
                        if metric_name
                        not in {
                            "cv_mae_mean",
                            "cv_mae_std",
                            "cv_rmse_mean",
                            "cv_rmse_std",
                            "cv_me_mean",
                            "cv_me_std",
                            "cv_r2_mean",
                            "cv_r2_std",
                        }
                    },
                }
            )

search_progress.close()
cv_experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if cv_experiment is None:
    raise ValueError(f"MLflow experiment {MLFLOW_EXPERIMENT_NAME!r} was not found")
current_cv_runs = mlflow.search_runs(
    experiment_ids=[cv_experiment.experiment_id],
    filter_string=(
        f"tags.execution_uuid = '{NOTEBOOK_EXECUTION_UUID}' "
        "and tags.phase = 'cv' "
        "and tags.run_type = 'candidate_parent'"
    ),
)
current_fold_runs = mlflow.search_runs(
    experiment_ids=[cv_experiment.experiment_id],
    filter_string=(
        f"tags.execution_uuid = '{NOTEBOOK_EXECUTION_UUID}' "
        "and tags.phase = 'cv' "
        "and tags.run_type = 'fold'"
    ),
)
parent_keys = {
    (
        str(row["tags.subset"]),
        int(float(row["params.max_depth"])),
        int(float(row["params.n_estimators"])),
        float(row["params.learning_rate"]),
        float(row["params.subsample"]),
        float(row["params.colsample_bytree"]),
        float(row["params.reg_lambda"]),
    )
    for _, row in current_cv_runs.iterrows()
}
if (
    len(current_cv_runs) != len(expected_candidate_keys)
    or parent_keys != expected_candidate_keys
):
    raise ValueError(
        "Current execution must produce the complete 192-candidate subset/hyperparameter set: "
        f"expected {len(expected_candidate_keys)} {sorted(expected_candidate_keys)}, "
        f"got {len(current_cv_runs)} {sorted(parent_keys)}"
    )
expected_fold_count = len(expected_candidate_keys) * N_VALIDATION_FOLDS
if len(current_fold_runs) != expected_fold_count:
    raise ValueError(
        f"Current execution must produce {expected_fold_count} nested fold runs, got {len(current_fold_runs)}"
    )
fold_keys = {
    (
        str(row["tags.subset"]),
        int(float(row["params.max_depth"])),
        int(float(row["params.n_estimators"])),
        float(row["params.learning_rate"]),
        float(row["params.subsample"]),
        float(row["params.colsample_bytree"]),
        float(row["params.reg_lambda"]),
        int(row["tags.fold"]),
    )
    for _, row in current_fold_runs.iterrows()
}
expected_fold_keys = {
    (*candidate_key, fold_number)
    for candidate_key in expected_candidate_keys
    for fold_number in range(1, N_VALIDATION_FOLDS + 1)
}
if fold_keys != expected_fold_keys:
    raise ValueError(
        "Current execution fold runs do not cover every candidate and fold"
    )
cv_results = pd.DataFrame(cv_results_rows)
if len(cv_results) != len(expected_candidate_keys):
    raise ValueError("The in-memory CV result table is incomplete")
if (
    set(
        zip(
            cv_results["subset"],
            cv_results["max_depth"],
            cv_results["n_estimators"],
            cv_results["learning_rate"],
            cv_results["subsample"],
            cv_results["colsample_bytree"],
            cv_results["reg_lambda"],
        )
    )
    != expected_candidate_keys
):
    raise ValueError("The in-memory CV result table does not match the candidate set")
cv_results = cv_results.sort_values(
    [
        "subset",
        "max_depth",
        "n_estimators",
        "learning_rate",
        "subsample",
        "colsample_bytree",
        "reg_lambda",
    ],
    kind="stable",
).reset_index(drop=True)
(
    selected_subset,
    selected_max_depth,
    selected_n_estimators,
    selected_learning_rate,
    selected_subsample,
    selected_colsample_bytree,
    selected_reg_lambda,
) = select_candidate(cv_results, CV_SELECTION_METRIC)
selected_feature_columns = FEATURE_SUBSETS[selected_subset]
selected_candidate_key = (
    selected_subset,
    selected_max_depth,
    selected_n_estimators,
    selected_learning_rate,
    selected_subsample,
    selected_colsample_bytree,
    selected_reg_lambda,
)
fold_horizon_rows = cv_horizon_rows_by_candidate[selected_candidate_key]
print(
    f"Selected XGBoost candidate by CV {CV_SELECTION_METRIC.upper()}: "
    f"{selected_subset!r}, max_depth={selected_max_depth}, n_estimators={selected_n_estimators}, "
    f"learning_rate={selected_learning_rate:g}, subsample={selected_subsample:g}, "
    f"colsample_bytree={selected_colsample_bytree:g}, reg_lambda={selected_reg_lambda:g}"
)
display(
    cv_results[
        [
            "subset",
            "feature_count",
            "max_depth",
            "n_estimators",
            "learning_rate",
            "subsample",
            "colsample_bytree",
            "reg_lambda",
            "mae_mean",
            "mae_std",
            "rmse_mean",
            "rmse_std",
            "me_mean",
            "me_std",
            "r2_mean",
            "r2_std",
        ]
    ]
)

## Retrain the selected subset and hyperparameters

The selected `(feature subset, hyperparameters)` candidate is retrained once on all eligible, chronologically ordered training rows, using the same `build_xgboost_estimator` construction path — and the same fixed `RANDOM_STATE` — as every CV fold. The estimator is fitted on the full eligible training cohort, then the sealed test predictors are scored. The model is held in memory here; it is written to disk together with its manifest only after the sealed-test cell below succeeds, so a crashed run leaves the previous artifacts intact.

In [ ]:
final_model = build_xgboost_estimator(
    max_depth=selected_max_depth,
    learning_rate=selected_learning_rate,
    n_estimators=selected_n_estimators,
    subsample=selected_subsample,
    colsample_bytree=selected_colsample_bytree,
    reg_lambda=selected_reg_lambda,
    n_jobs=XGBOOST_N_JOBS,
    random_state=RANDOM_STATE,
)
final_model.fit(
    numeric_predictors(train_rows, selected_feature_columns),
    train_rows[TARGET_COLUMNS],
)
test_predictions = validate_predictions(
    final_model.predict(numeric_predictors(test_rows, selected_feature_columns)),
    expected_rows=len(test_rows),
    target_columns=TARGET_COLUMNS,
    artifact_name="test",
)
print(
    f"Fitted the selected XGBoost candidate on {len(train_rows):,} eligible training "
    f"rows and scored {len(test_rows):,} sealed-test rows."
)

## Evaluate on the test cohort

A single scoring pass over the sealed test cohort reports aggregate MAE/RMSE, the same metrics for each configured lead in the direct forecast, and a short preview for comparison with actual targets. Plot and MLflow labels identify the selected subset and hyperparameters. There is no second pass and no refitting.

The model and its manifest are written at the end of this cell. The manifest is the durable record of this execution — including its schema version, selected subset and hyperparameters, complete realized CV candidate table, aggregate sealed-test metrics, and per-horizon sealed-test metrics — and it is what the evaluation section below reads. MLflow still receives the same params, metrics, and figures; it remains the run log and UI, not the record the notebook reads back.

In [ ]:
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS],
    test_predictions,
    target_columns=TARGET_COLUMNS,
    station_id=station_id,
)
if not np.isfinite(aggregate_metrics[["mae", "rmse", "me", "r2"]].to_numpy()).all():
    raise ValueError("XGBoost reported non-finite aggregate metrics")
if not np.isfinite(per_horizon_metrics[["mae", "rmse", "me", "r2"]].to_numpy()).all():
    raise ValueError("XGBoost reported non-finite horizon metrics")
with mlflow.start_run(
    run_name=f"xgboost_test_{selected_subset}",
    nested=False,
    tags={
        "phase": "test",
        "run_type": "sealed_test",
        "subset": selected_subset,
        "execution_uuid": NOTEBOOK_EXECUTION_UUID,
    },
):
    mlflow.log_params(
        {
            "phase": "test",
            "run_type": "sealed_test",
            "subset": selected_subset,
            "feature_count": len(selected_feature_columns),
            "feature_columns": json.dumps(selected_feature_columns),
            **INPUT_PARQUET_SHA256_PARAMS,
            "max_depth": selected_max_depth,
            "n_estimators": selected_n_estimators,
            "learning_rate": selected_learning_rate,
            "subsample": selected_subsample,
            "colsample_bytree": selected_colsample_bytree,
            "reg_lambda": selected_reg_lambda,
            "selection_metric": CV_SELECTION_METRIC,
            "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
            "cv_selected_metric": float(
                cv_results.loc[
                    cv_results["subset"].eq(selected_subset)
                    & cv_results["max_depth"].eq(selected_max_depth)
                    & cv_results["n_estimators"].eq(selected_n_estimators)
                    & cv_results["learning_rate"].eq(selected_learning_rate)
                    & cv_results["subsample"].eq(selected_subsample)
                    & cv_results["colsample_bytree"].eq(selected_colsample_bytree)
                    & cv_results["reg_lambda"].eq(selected_reg_lambda),
                    f"{CV_SELECTION_METRIC}_mean",
                ].iloc[0]
            ),
            "scored_issue_times": len(test_rows),
        }
    )
    mlflow.log_metrics(
        {
            "test_mae": float(aggregate_metrics.iloc[0]["mae"]),
            "test_rmse": float(aggregate_metrics.iloc[0]["rmse"]),
            "test_me": float(aggregate_metrics.iloc[0]["me"]),
            "test_r2": float(aggregate_metrics.iloc[0]["r2"]),
            **{
                f"test_mae_horizon_{row.horizon_hours:02d}": float(row.mae)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_me_horizon_{row.horizon_hours:02d}": float(row.me)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_r2_horizon_{row.horizon_hours:02d}": float(row.r2)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_rmse_horizon_{row.horizon_hours:02d}": float(row.rmse)
                for row in per_horizon_metrics.itertuples()
            },
        }
    )
    cv_horizon_metrics = pd.concat(fold_horizon_rows, ignore_index=True)
    cv_rmse_mae_boxplots_fig = cv_error_boxplots_figure(
        cv_horizon_metrics,
        TARGET_COLUMNS,
        title=f"XGBoost CV errors — {selected_subset}, max_depth={selected_max_depth}, n_estimators={selected_n_estimators}",
    )
    mlflow.log_figure(cv_rmse_mae_boxplots_fig, "cv_rmse_mae_boxplots.png")
    plt.show()
    plt.close(cv_rmse_mae_boxplots_fig)
    test_error_boxplots_fig = test_error_boxplots_figure(
        test_rows,
        test_predictions,
        per_horizon_metrics,
        TARGET_COLUMNS,
        title=f"XGBoost final-test errors — {selected_subset}, max_depth={selected_max_depth}, n_estimators={selected_n_estimators}",
    )
    mlflow.log_figure(test_error_boxplots_fig, "test_error_boxplots.png")
    plt.show()
    plt.close(test_error_boxplots_fig)
    test_predicted_vs_actual_fig = predicted_vs_actual_figure(
        test_rows[TARGET_COLUMNS],
        test_predictions,
        TARGET_COLUMNS,
        title=f"XGBoost predicted vs actual — {selected_subset}, max_depth={selected_max_depth}, n_estimators={selected_n_estimators}",
    )
    mlflow.log_figure(test_predicted_vs_actual_fig, "test_predicted_vs_actual.png")
    plt.show()
    plt.close(test_predicted_vs_actual_fig)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
dump(final_model, MODEL_PATH)
save_xgboost_manifest(
    MODEL_METADATA_PATH,
    model_path=MODEL_PATH,
    execution_uuid=NOTEBOOK_EXECUTION_UUID,
    contract=contract,
    feature_subsets=FEATURE_SUBSETS,
    selected_subset=selected_subset,
    selected_max_depth=selected_max_depth,
    selected_n_estimators=selected_n_estimators,
    selected_learning_rate=selected_learning_rate,
    selected_subsample=selected_subsample,
    selected_colsample_bytree=selected_colsample_bytree,
    selected_reg_lambda=selected_reg_lambda,
    selection_metric=CV_SELECTION_METRIC,
    cv_results=cv_results,
    sealed_test_metrics={
        "test_mae": float(aggregate_metrics.iloc[0]["mae"]),
        "test_rmse": float(aggregate_metrics.iloc[0]["rmse"]),
        "test_me": float(aggregate_metrics.iloc[0]["me"]),
        "test_r2": float(aggregate_metrics.iloc[0]["r2"]),
    },
    per_horizon_metrics=per_horizon_metrics,
    cohort={
        "contract": "full_feature_columns",
        "rule": "target_valid and complete full predictor and target contract",
        "train_raw_rows": dataset.raw_row_counts["train"],
        "train_eligible_rows": len(train_rows),
        "test_raw_rows": dataset.raw_row_counts["test"],
        "test_eligible_rows": len(test_rows),
        "same_folds_for_all_candidates": True,
    },
    training={
        "source_artifact": str(train_path),
        "raw_rows": dataset.raw_row_counts["train"],
        "eligible_rows": len(train_rows),
        "eligibility": "target_valid and complete full predictor and target contract",
        "timestamp_start": train_rows["timestamp"].iloc[0].isoformat(),
        "timestamp_end": train_rows["timestamp"].iloc[-1].isoformat(),
    },
)
print(f"Saved XGBoost model to {MODEL_PATH}")
print(f"Saved XGBoost model manifest to {MODEL_METADATA_PATH}")
print(
    f"XGBoost test results for {station_id} "
    f"(selected subset={selected_subset!r}, max_depth={selected_max_depth}, "
    f"n_estimators={selected_n_estimators}, learning_rate={selected_learning_rate:g}, "
    f"subsample={selected_subsample:g}, colsample_bytree={selected_colsample_bytree:g}, "
    f"reg_lambda={selected_reg_lambda:g})"
)
display(aggregate_metrics)
display(per_horizon_metrics)
display(
    prediction_preview(
        test_rows,
        test_predictions,
        target_columns=TARGET_COLUMNS,
    ).head(PREDICTION_PREVIEW_ROWS)
)

# XGBoost saved-model evaluation

This read-only section reads the saved XGBoost manifest, `models/xgboost_{TARGET_STATION_ID}.json`, which the sealed-test cell above writes once per successful execution. It compares that execution's feature-subset/hyperparameter candidates using their cross-validation metrics and reports sealed-test metrics only for the selected candidate. It does not query MLflow: the manifest is the record, MLflow is the run log.

## Load the saved XGBoost execution record

The joined dataset and the manifest are loaded here, so this section does not depend on any variable created by the training cells and can be re-run on its own in a fresh kernel. `load_xgboost_manifest()` checks the manifest against the *current* feature contract — station, horizon, full predictor columns, target columns, and the selected subset's columns — so a Stage-3 re-run that changes the contract fails here instead of silently scoring a stale model.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.dataset import load_joined_dataset
from src.plots import forecast_window_figures
from src.xgboost_model import load_xgboost_manifest, score_saved_model

if CV_SELECTION_METRIC not in {"mae", "rmse"}:
    raise ValueError("CV_SELECTION_METRIC must be either 'mae' or 'rmse'")

COMPARISON_PROCESSED_DIR = Path("data/processed/joined")
COMPARISON_METADATA_PATH = (
    COMPARISON_PROCESSED_DIR / "all_stations_feature_metadata.json"
)
COMPARISON_TRAIN_PATH = COMPARISON_PROCESSED_DIR / "all_stations_train_features.parquet"
COMPARISON_TEST_PATH = COMPARISON_PROCESSED_DIR / "all_stations_test_features.parquet"
COMPARISON_MODEL_PATH = Path("models") / f"xgboost_{TARGET_STATION_ID}.joblib"
COMPARISON_MODEL_METADATA_PATH = Path("models") / f"xgboost_{TARGET_STATION_ID}.json"

comparison_dataset = load_joined_dataset(
    COMPARISON_METADATA_PATH,
    COMPARISON_TRAIN_PATH,
    COMPARISON_TEST_PATH,
    station_id=TARGET_STATION_ID,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
comparison_contract = comparison_dataset.contract
COMPARISON_TARGET_COLUMNS = list(comparison_contract.target_columns)
comparison_test_rows = comparison_dataset.test_rows

xgboost_manifest = load_xgboost_manifest(
    COMPARISON_MODEL_METADATA_PATH,
    contract=comparison_contract,
    feature_subsets=comparison_dataset.feature_subsets,
)

## Inspect the recorded execution

The manifest is written once, after the sealed test has been scored, so an execution that crashed part-way leaves no record and the previous manifest survives untouched. The `execution_uuid` below is the same tag the MLflow runs of that execution carry, which is how the two views are tied together.

In [ ]:
selected_candidate_table = xgboost_manifest.cv_results
selected_horizon_metrics = xgboost_manifest.horizon_metrics
selected_candidate = selected_candidate_table.loc[
    selected_candidate_table["subset"].eq(xgboost_manifest.selected_subset)
    & selected_candidate_table["max_depth"].eq(xgboost_manifest.selected_max_depth)
    & selected_candidate_table["n_estimators"].eq(
        xgboost_manifest.selected_n_estimators
    )
    & selected_candidate_table["learning_rate"].eq(
        xgboost_manifest.selected_learning_rate
    )
    & selected_candidate_table["subsample"].eq(xgboost_manifest.selected_subsample)
    & selected_candidate_table["colsample_bytree"].eq(
        xgboost_manifest.selected_colsample_bytree
    )
    & selected_candidate_table["reg_lambda"].eq(xgboost_manifest.selected_reg_lambda)
].iloc[0]
selected_execution_summary = pd.DataFrame(
    [
        {
            "execution_uuid": xgboost_manifest.execution_uuid,
            "manifest": str(COMPARISON_MODEL_METADATA_PATH),
            "candidate_count": len(selected_candidate_table),
            "selection_metric": xgboost_manifest.selection_metric,
            "selected_subset": xgboost_manifest.selected_subset,
            "selected_max_depth": xgboost_manifest.selected_max_depth,
            "selected_n_estimators": xgboost_manifest.selected_n_estimators,
            "selected_learning_rate": xgboost_manifest.selected_learning_rate,
            "selected_subsample": xgboost_manifest.selected_subsample,
            "selected_colsample_bytree": xgboost_manifest.selected_colsample_bytree,
            "selected_reg_lambda": xgboost_manifest.selected_reg_lambda,
        }
    ]
)
display(selected_execution_summary)

## Compare cross-validation candidates

Candidate ranking uses only the recorded CV metrics and the configured selection metric; the sealed-test metrics are not used to rank candidates.

In [ ]:
candidate_columns = [
    "subset",
    "feature_count",
    "max_depth",
    "n_estimators",
    "learning_rate",
    "subsample",
    "colsample_bytree",
    "reg_lambda",
    "mae_mean",
    "mae_std",
    "rmse_mean",
    "rmse_std",
    "me_mean",
    "me_std",
    "r2_mean",
    "r2_std",
]
candidate_comparison_table = selected_candidate_table[candidate_columns].copy()
display(candidate_comparison_table)

## Visualize cross-validation error

The heatmap shows CV RMSE across feature subsets and max_depth values, faceted by n_estimators. The line chart adds fold-to-fold RMSE variation as error bars, with learning_rate on a log scale.

In [ ]:
N_ESTIMATORS_FACETS = sorted(candidate_comparison_table["n_estimators"].unique())
cv_rmse_heatmap_zmin = candidate_comparison_table["rmse_mean"].min()
cv_rmse_heatmap_zmax = candidate_comparison_table["rmse_mean"].max()
cv_rmse_heatmap_figure = make_subplots(
    rows=1,
    cols=len(N_ESTIMATORS_FACETS),
    subplot_titles=[f"n_estimators={value}" for value in N_ESTIMATORS_FACETS],
    shared_yaxes=True,
)
for column_index, n_estimators_value in enumerate(N_ESTIMATORS_FACETS, start=1):
    facet_table = candidate_comparison_table[
        candidate_comparison_table["n_estimators"].eq(n_estimators_value)
    ]
    facet_heatmap_values = (
        facet_table.groupby(["subset", "max_depth"])["rmse_mean"]
        .mean()
        .unstack("max_depth")
        .sort_index(axis=0)
        .sort_index(axis=1)
    )
    cv_rmse_heatmap_figure.add_trace(
        go.Heatmap(
            z=facet_heatmap_values.to_numpy(),
            x=list(map(str, facet_heatmap_values.columns.tolist())),
            y=facet_heatmap_values.index.tolist(),
            zmin=cv_rmse_heatmap_zmin,
            zmax=cv_rmse_heatmap_zmax,
            coloraxis="coloraxis",
            hovertemplate="Subset=%{y}<br>max_depth=%{x}<br>CV RMSE=%{z:.4f}<extra></extra>",
        ),
        row=1,
        col=column_index,
    )
cv_rmse_heatmap_figure.update_layout(
    title="XGBoost candidate CV RMSE by feature subset, max_depth, and n_estimators (mean over other sampled hyperparameters)",
    coloraxis={"colorscale": "Viridis", "colorbar": {"title": "CV RMSE"}},
)
cv_rmse_heatmap_figure.update_xaxes(title_text="max_depth")
cv_rmse_heatmap_figure.update_yaxes(title_text="Feature subset", col=1)
display(cv_rmse_heatmap_figure)

In [ ]:
cv_rmse_by_learning_rate_figure = go.Figure()
for subset_name, subset_candidates in candidate_comparison_table.groupby(
    "subset", sort=True
):
    subset_summary = (
        subset_candidates.groupby("learning_rate")["rmse_mean"]
        .agg(["mean", "std"])
        .sort_index()
    )
    cv_rmse_by_learning_rate_figure.add_trace(
        go.Scatter(
            x=subset_summary.index,
            y=subset_summary["mean"],
            mode="lines+markers",
            name=subset_name,
            error_y={
                "type": "data",
                "array": subset_summary["std"].fillna(0.0),
                "visible": True,
            },
        )
    )
cv_rmse_by_learning_rate_figure.update_layout(
    title="XGBoost CV RMSE versus learning_rate, by feature subset (mean over other sampled hyperparameters)",
    xaxis_title="learning_rate",
    yaxis_title="CV RMSE",
    xaxis_type="log",
)
display(cv_rmse_by_learning_rate_figure)

## Inspect selected-candidate sealed-test performance

These values belong only to the candidate recorded by the selected sealed-test run. The final chart shows its MAE and RMSE at each available forecast horizon.

In [ ]:
selected_candidate_sealed_test_summary = pd.DataFrame(
    [
        {
            "execution_uuid": xgboost_manifest.execution_uuid,
            "subset": selected_candidate["subset"],
            "feature_count": selected_candidate["feature_count"],
            "max_depth": selected_candidate["max_depth"],
            "n_estimators": selected_candidate["n_estimators"],
            "learning_rate": selected_candidate["learning_rate"],
            "subsample": selected_candidate["subsample"],
            "colsample_bytree": selected_candidate["colsample_bytree"],
            "reg_lambda": selected_candidate["reg_lambda"],
            "cv_mae_mean": selected_candidate["mae_mean"],
            "cv_rmse_mean": selected_candidate["rmse_mean"],
            "cv_me_mean": selected_candidate["me_mean"],
            "cv_r2_mean": selected_candidate["r2_mean"],
            **xgboost_manifest.sealed_test_metrics,
        }
    ]
)
display(selected_candidate_sealed_test_summary)

sealed_test_horizon_figure = go.Figure(
    [
        go.Scatter(
            x=selected_horizon_metrics["horizon_hours"],
            y=selected_horizon_metrics[metric_name],
            mode="lines+markers",
            name=metric_name.upper(),
        )
        for metric_name in ("test_mae", "test_rmse", "test_me", "test_r2")
    ]
)
sealed_test_horizon_figure.update_layout(
    title="Selected XGBoost candidate sealed-test error by horizon",
    xaxis_title="Forecast horizon (hours)",
    yaxis_title="Error",
)
display(sealed_test_horizon_figure)

## Reload and score the saved XGBoost model

Reloads the saved XGBoost model and scores the eligible sealed-test cohort on the manifest's selected feature columns, without retraining or changing the stored prediction semantics.

In [ ]:
comparison_prediction_values = score_saved_model(
    xgboost_manifest,
    COMPARISON_MODEL_PATH,
    comparison_test_rows,
)
print(
    f"Scored {len(comparison_test_rows):,} eligible sealed-test rows with "
    f"{len(COMPARISON_TARGET_COLUMNS)} horizons using the saved XGBoost model."
)

In [ ]:
comparison_prediction_columns = [
    f"prediction_{target_column}" for target_column in COMPARISON_TARGET_COLUMNS
]
comparison_prediction_table = (
    comparison_test_rows[["timestamp", *COMPARISON_TARGET_COLUMNS]]
    .reset_index(drop=True)
    .rename(columns={"timestamp": "issue_time"})
)
comparison_prediction_table = pd.concat(
    [
        comparison_prediction_table,
        pd.DataFrame(
            comparison_prediction_values,
            columns=comparison_prediction_columns,
        ),
    ],
    axis=1,
)
comparison_prediction_table["issue_time"] = pd.to_datetime(
    comparison_prediction_table["issue_time"], utc=True
)

comparison_issue_times = comparison_prediction_table["issue_time"]
comparison_horizons = list(range(1, FORECAST_HORIZON_HOURS + 1))
comparison_horizon_labels = [f"H+{horizon:02d}" for horizon in comparison_horizons]
comparison_time_series_frames = []
for horizon in comparison_horizons:
    target_column = COMPARISON_TARGET_COLUMNS[horizon - 1]
    prediction_column = comparison_prediction_columns[horizon - 1]
    valid_times = comparison_issue_times + pd.to_timedelta(horizon, unit="h")
    comparison_time_series_frames.append(
        go.Frame(
            name=comparison_horizon_labels[horizon - 1],
            data=[
                go.Scattergl(
                    x=valid_times,
                    y=comparison_prediction_table[target_column],
                    customdata=comparison_issue_times,
                    mode="lines+markers",
                    name="Actual",
                    hovertemplate="Valid time=%{x}<br>Issue time=%{customdata}<br>Actual=%{y:.3f}<extra></extra>",
                ),
                go.Scattergl(
                    x=valid_times,
                    y=comparison_prediction_table[prediction_column],
                    customdata=comparison_issue_times,
                    mode="lines+markers",
                    name="Prediction",
                    hovertemplate="Valid time=%{x}<br>Issue time=%{customdata}<br>Prediction=%{y:.3f}<extra></extra>",
                ),
            ],
        )
    )
comparison_time_series_steps = [
    {
        "label": comparison_horizon_labels[horizon - 1],
        "method": "animate",
        "args": [[comparison_horizon_labels[horizon - 1]], {"mode": "immediate"}],
    }
    for horizon in comparison_horizons
]
comparison_time_series_figure = go.Figure(
    data=comparison_time_series_frames[0].data,
    frames=comparison_time_series_frames,
    layout={
        "title": "Saved XGBoost predictions across forecast horizons",
        "xaxis_title": "Valid time",
        "yaxis_title": "Water level",
        "hovermode": "x unified",
        "sliders": [
            {
                "active": 0,
                "currentvalue": {"prefix": "Forecast horizon: "},
                "steps": comparison_time_series_steps,
            }
        ],
    },
)
xgboost_prediction_time_series_figure = comparison_time_series_figure
display(comparison_time_series_figure)

## Inspect best and worst XGBoost forecast windows

The following plots use the saved-model predictions and select sealed-test issue times by the RMSE calculated across every manifest-declared forecast horizon. A context window is eligible only when the target-station water-level series contains every hourly observation across the notebook's configured context window, with no imputed observations.

In [ ]:
xgboost_forecast_window_figures = forecast_window_figures(
    comparison_prediction_table,
    comparison_dataset.target_context_series,
    water_level_column=f"{TARGET_STATION_ID}__water_level",
    imputed_column=f"{TARGET_STATION_ID}__imputed",
    prediction_columns=comparison_prediction_columns,
    target_columns=COMPARISON_TARGET_COLUMNS,
    horizons=comparison_horizons,
    label_prefix="XGBoost",
)

### Best

In [ ]:
best_xgboost_forecast_window_figure = xgboost_forecast_window_figures["best"]
display(best_xgboost_forecast_window_figure)

### Worst

In [ ]:
worst_xgboost_forecast_window_figure = xgboost_forecast_window_figures["worst"]
display(worst_xgboost_forecast_window_figure)

## Compare absolute and signed errors

Each box contains all eligible sealed-test errors for one horizon. Absolute-error markers reuse the manifest's per-horizon MAE/RMSE values; signed errors follow the convention `prediction - actual`.

In [ ]:
comparison_actual_values = comparison_test_rows[COMPARISON_TARGET_COLUMNS].to_numpy(
    dtype=float
)
signed_errors = comparison_prediction_values - comparison_actual_values
absolute_errors = np.abs(signed_errors)

absolute_error_boxplot_figure = go.Figure(
    data=[
        go.Box(
            x=comparison_horizon_labels * len(absolute_errors),
            y=absolute_errors.reshape(-1),
            name="Boxplots",
            boxpoints=False,
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=selected_horizon_metrics["test_mae"],
            mode="markers",
            name="MAE",
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=selected_horizon_metrics["test_rmse"],
            mode="markers",
            name="RMSE",
        ),
    ]
)
absolute_error_boxplot_figure.update_layout(
    title="Saved XGBoost absolute errors by forecast horizon",
    xaxis={
        "title": "Forecast horizon",
        "type": "category",
        "categoryorder": "array",
        "categoryarray": comparison_horizon_labels,
    },
    yaxis_title="Absolute error",
)

display(absolute_error_boxplot_figure)

In [ ]:
signed_error_boxplot_figure = go.Figure(
    data=[
        go.Box(
            x=comparison_horizon_labels * len(signed_errors),
            y=signed_errors.reshape(-1),
            name="Boxplots",
            boxpoints=False,
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=signed_errors.mean(axis=0),
            mode="markers",
            name="Mean error",
        ),
    ]
)
signed_error_boxplot_figure.update_layout(
    title="Saved XGBoost signed errors by forecast horizon",
    xaxis={
        "title": "Forecast horizon",
        "type": "category",
        "categoryorder": "array",
        "categoryarray": comparison_horizon_labels,
    },
    yaxis_title="Signed error (prediction - actual)",
)
signed_error_boxplot_figure.add_hline(
    y=0,
    line_dash="dash",
    line_color="black",
)
display(signed_error_boxplot_figure)

## Selected XGBoost model

Ridge's final section extracts a closed-form linear coefficient formula and MLP's reports its fitted architecture; neither has a counterpart for a tree ensemble. Instead, this section reloads the saved model and reports its gain-based feature importances plus the selected hyperparameters and fitted tree count.

In [ ]:
from IPython.display import Markdown
from joblib import load as load_joblib

importance_model = load_joblib(COMPARISON_MODEL_PATH)
importance_scores = importance_model.get_booster().get_score(importance_type="gain")
importance_table = (
    pd.DataFrame(
        {
            "feature": list(importance_scores.keys()),
            "gain": list(importance_scores.values()),
        }
    )
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)

feature_importance_figure = go.Figure(
    go.Bar(
        x=importance_table["gain"],
        y=importance_table["feature"],
        orientation="h",
    )
)
feature_importance_figure.update_layout(
    title="Saved XGBoost gain-based feature importance",
    xaxis_title="Total gain",
    yaxis_title="Feature",
    yaxis={"categoryorder": "total ascending"},
    height=max(400, 20 * len(importance_table)),
)
display(feature_importance_figure)

architecture_summary_table = pd.DataFrame(
    [
        {
            "max_depth": xgboost_manifest.selected_max_depth,
            "n_estimators": xgboost_manifest.selected_n_estimators,
            "learning_rate": xgboost_manifest.selected_learning_rate,
            "subsample": xgboost_manifest.selected_subsample,
            "colsample_bytree": xgboost_manifest.selected_colsample_bytree,
            "reg_lambda": xgboost_manifest.selected_reg_lambda,
            "boosting_rounds": importance_model.get_booster().num_boosted_rounds(),
            "fitted_trees": len(importance_model.get_booster().get_dump()),
        }
    ]
)
display(
    Markdown(
        f"""The saved model uses subset **{xgboost_manifest.selected_subset}**,
max_depth **{xgboost_manifest.selected_max_depth}**, n_estimators
**{xgboost_manifest.selected_n_estimators}**, learning_rate
**{xgboost_manifest.selected_learning_rate:g}**."""
    )
)
display(architecture_summary_table)